# Pandas : cours expliqué et exercices progressifs

Ce support reprend **tout le contenu pédagogique de `03_pandas_empty_v2.ipynb` (134 cellules)**, dans le même ordre : création, exploration, sélection, modification, nettoyage, assemblage, statistiques, fonctions et application au compteur électrique. Le fichier original reste intact.

**Mode d’emploi :** choisis le noyau Python du projet (`.venv`), exécute les imports ci-dessous, puis avance de haut en bas. Chaque partie contient une explication, un exemple exécutable, des exercices et une cellule pour ta réponse. Les corrigés se déplient sous les exercices ; tu peux les copier dans ta cellule pour les exécuter. Les exemples utilisent des données distinctes autant que possible. Pour repartir de zéro, redémarre le noyau et exécute les exemples dans l’ordre.

Les indications de résultat servent à vérifier ton raisonnement. Essaie de prédire les résultats avant d’exécuter le code, puis explique chaque ligne avec tes propres mots.

**Quelques précisions par rapport à l’original :** `info()` décrit la structure, `describe()` résume les valeurs ; `apply()` ne renvoie pas toujours un DataFrame et n’est pas automatiquement plus rapide ; l’exemple aléatoire original n’a pas d’index temporel explicite. La dernière partie utilise des variables non définies : ici elle est complétée et expliquée. Le texte y alterne entre production et consommation constante ; nous choisissons une **production supplémentaire de 50 W**.


## Parcours

1. [Imports, Series et valeurs manquantes](#notion-1)
2. [Créer un DataFrame et comprendre les types](#notion-2)
3. [Explorer : dtypes, info, describe, head et tail](#notion-3)
4. [Sélectionner des colonnes](#notion-4)
5. [Sélection par étiquette : loc et at](#notion-5)
6. [Sélection par position : iloc et iat](#notion-6)
7. [Filtrer : masques booléens et isin](#notion-7)
8. [Modifier des cellules et créer des colonnes](#notion-8)
9. [Comprendre l’alignement des Series](#notion-9)
10. [Définir, réinitialiser et réindexer un index](#notion-10)
11. [Valeurs manquantes : détecter, supprimer, remplir](#notion-11)
12. [Détecter et supprimer les doublons](#notion-12)
13. [Concaténer des tableaux : concat, axis et join](#notion-13)
14. [Joindre sur une clé : merge et multiplicité](#notion-14)
15. [Statistiques et sens des axes](#notion-15)
16. [Fonctions, apply, map et opérations vectorisées](#notion-16)
17. [Lambda, modulo et dictionnaires de correspondance](#notion-17)
18. [Lire le CSV et contrôler les données](#notion-18)
19. [Dates, fuseaux horaires et pas de temps](#notion-19)
20. [Autoconsommation : minimum et taux global](#notion-20)
21. [Tracer des courbes avec Plotly](#notion-21)
22. [date_range et conversion d’une production de 50 W](#notion-22)
23. [Projet de synthèse : joindre et recalculer avec l’hydro](#notion-23)
24. [Pour aller plus loin : groupby et tableau croisé](#notion-24)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

print("Version de pandas :", pd.__version__)


<a id="notion-1"></a>

## 1. Imports, Series et valeurs manquantes

**Repère dans le notebook original : cellules 1–5** (numérotation à partir de 1, textes compris).


`import pandas as pd` donne le raccourci `pd` à la bibliothèque Pandas. NumPy (`np`) fournit notamment des tableaux numériques et `np.nan`.

Une **Series** est une suite de valeurs avec une étiquette par valeur : son **index**. Sans index explicite, Pandas crée `0, 1, 2…`. `np.nan` signifie « valeur manquante », pas zéro. Sa présence peut faire passer une série d’entiers au type flottant. Un **DataFrame** est un tableau à deux dimensions composé de colonnes, chacune accessible comme une Series.


In [ ]:
s_notes = pd.Series([12, 15, np.nan, 18], index=["Alice", "Benoît", "Chloé", "David"])
display(s_notes)
print(s_notes.dtype)


### À toi de jouer


1. Crée `temperatures` contenant `8, 11, np.nan, 14`, avec les étiquettes `lun, mar, mer, jeu`.
2. Affiche son index et son type.
3. Calcule sa moyenne. Pourquoi vaut-elle **11** ? Que se passerait-il si la valeur manquante était remplacée par zéro ?


In [ ]:
temperatures = pd.Series([8, 11, np.nan, 14], index=["lun", "mar", "mer","jeu"])
display(temperatures)
print(temperatures.dtype)

print(np.mean(temperatures))
print(temperatures.fillna(0).mean())

<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
temperatures = pd.Series([8, 11, np.nan, 14], index=["lun", "mar", "mer", "jeu"])
print(temperatures.index, temperatures.dtype)
print(temperatures.mean())  # (8 + 11 + 14) / 3 = 11 ; NaN est ignoré.
print(temperatures.fillna(0).mean())  # 8.25 : zéro est une vraie mesure.
```

</details>


<a id="notion-2"></a>

## 2. Créer un DataFrame et comprendre les types

**Repère dans le notebook original : cellules 6–10** (numérotation à partir de 1, textes compris).


Un dictionnaire associe **nom de colonne → données**. Les listes doivent avoir une longueur compatible ; un scalaire, comme `1.0`, est répété sur toutes les lignes. `index=` nomme les lignes ; `columns=` nomme les colonnes d’un tableau NumPy.

`pd.Timestamp` représente une date/heure ; `pd.Categorical` représente un ensemble de catégories ; `dtype="int32"` demande des entiers sur 32 bits. `range(4)` produit les entiers de 0 à 3. `np.arange(12).reshape(3, 4)` réorganise 12 nombres en 3 lignes et 4 colonnes. `np.random.randn(…)`, utilisé dans l’original, génère des valeurs aléatoires normales ; ici un générateur avec graine permet de reproduire les résultats.

L’autocomplétion peut aider à explorer `df.`, mais `df["colonne"]` fonctionne aussi pour les noms avec des espaces ou en conflit avec une méthode.


In [ ]:
rng = np.random.default_rng(42)
display(pd.DataFrame(rng.standard_normal((3, 2)), columns=["A", "B"]))
df_types = pd.DataFrame({
    "mesure": 1.0,
    "date": pd.Timestamp("2023-08-01"),
    "numero": range(4),
    "entier": np.array([3] * 4, dtype="int32"),
    "groupe": pd.Categorical(["test", "train", "test", "train"]),
    "texte": "foo",
}, index=["One", "Two", "Three", "Four"])
display(df_types)


### À toi de jouer


1. Crée `capteurs` indexé par `C1, C2, C3`, avec `temperature=[18.5, 20.0, 17.5]`, `actif=True` et `zone` catégorielle valant `nord, sud, nord`.
2. Ajoute une colonne `date` contenant le 1er janvier 2024 pour chaque ligne.
3. Crée un autre DataFrame de 3 lignes × 4 colonnes avec les nombres de 0 à 11 et les colonnes `A, B, C, D`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
capteurs = pd.DataFrame({
    "temperature": [18.5, 20.0, 17.5], "actif": True,
    "zone": pd.Categorical(["nord", "sud", "nord"]),
}, index=["C1", "C2", "C3"])
capteurs["date"] = pd.Timestamp("2024-01-01")
display(capteurs)
display(pd.DataFrame(np.arange(12).reshape(3, 4), columns=list("ABCD")))
```

</details>


<a id="notion-3"></a>

## 3. Explorer : dtypes, info, describe, head et tail

**Repère dans le notebook original : cellules 11–21** (numérotation à partir de 1, textes compris).


Avant de calculer, vérifie la structure et un échantillon des données. `dtypes` donne le type de chaque colonne ; `info()` affiche les types, les effectifs non manquants et la mémoire utilisée (elle renvoie `None`). `describe()` donne notamment `count`, moyenne, écart-type, minimum, quartiles et maximum des colonnes numériques ; les dates peuvent aussi être résumées selon les types présents. `describe(include="all")` étend le résumé aux autres colonnes.

`head(n)` montre les premières lignes ; `tail(n)` les dernières (5 par défaut). `index` contient les étiquettes de lignes, `columns` celles des colonnes. `shape` donne `(nombre de lignes, nombre de colonnes)`.


In [ ]:
df_vue = pd.DataFrame({"nom": ["A", "B", "C", "D"], "score": [10, 14, np.nan, 18]})
display(df_vue.dtypes)
df_vue.info()
display(df_vue.describe())
display(df_vue.head(2), df_vue.tail(1))
print(df_vue.index, df_vue.columns, df_vue.shape)


### À toi de jouer


1. Affiche les 2 dernières lignes de `df_vue` et les types.
2. Combien de scores sont renseignés ? Quelle est leur moyenne ?
3. Affiche un résumé incluant la colonne `nom` et interprète `unique`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(df_vue.tail(2), df_vue.dtypes)
print(df_vue["score"].count())  # 3 scores, pour 4 lignes.
print(df_vue["score"].mean())  # 14.
display(df_vue.describe(include="all"))  # unique=4 noms distincts.
```

</details>


<a id="notion-4"></a>

## 4. Sélectionner des colonnes

**Repère dans le notebook original : cellules 22–25** (numérotation à partir de 1, textes compris).


`df["A"]` renvoie une **Series** ; `df[["A"]]` renvoie un **DataFrame** à une colonne. Les doubles crochets signifient que l’on transmet une liste de noms au DataFrame. Avec plusieurs noms, leur ordre détermine l’ordre des colonnes du résultat. Sélectionner n’efface pas les autres colonnes du tableau d’origine.


In [ ]:
df_selection = pd.DataFrame({"A": [10, 20, 30, 40], "B": [1, 2, 3, 4], "C": [0, 1, 2, 3]},
                            index=["One", "Two", "Three", "Four"])
display(df_selection["A"], df_selection[["A", "B"]])


### À toi de jouer


1. Sélectionne `C` comme Series, puis comme DataFrame.
2. Sélectionne `B` puis `A`, dans cet ordre.
3. Vérifie les types avec `type(...)` et les dimensions avec `.shape`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
c_serie = df_selection["C"]
c_table = df_selection[["C"]]
display(df_selection[["B", "A"]])
print(type(c_serie), c_serie.shape)  # Series, (4,)
print(type(c_table), c_table.shape)  # DataFrame, (4, 1)
```

</details>


<a id="notion-5"></a>

## 5. Sélection par étiquette : loc et at

**Repère dans le notebook original : cellules 26–33** (numérotation à partir de 1, textes compris).


`.loc[lignes, colonnes]` utilise les **étiquettes**. Une liste choisit plusieurs étiquettes ; `:` sélectionne une tranche ou tout un axe. Dans une tranche `.loc["One":"Three"]`, les deux bornes sont incluses pour cet index unique. Ce sont les positions dans l’index existant qui définissent ici la tranche, pas un tri alphabétique.

`.at[ligne, colonne]` accède à une seule valeur lorsque les étiquettes sont uniques. Les paramètres sont toujours dans l’ordre **ligne, colonne**.


In [ ]:
display(df_selection.loc["Two"])
display(df_selection.loc["One":"Three", "A":"B"])
print(df_selection.at["Two", "A"])


### À toi de jouer


1. Extrais les lignes `Two` et `Four`, colonnes `A` et `C`.
2. Extrais toutes les colonnes des lignes `Two` à `Four` incluses.
3. Récupère la valeur de `C` à la ligne `Three` avec `.at` (attendu : **2**).


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(df_selection.loc[["Two", "Four"], ["A", "C"]])
display(df_selection.loc["Two":"Four", :])
print(df_selection.at["Three", "C"])
```

</details>


<a id="notion-6"></a>

## 6. Sélection par position : iloc et iat

**Repère dans le notebook original : cellules 34–41** (numérotation à partir de 1, textes compris).


`.iloc` utilise les **positions entières à partir de 0**, indépendamment des étiquettes. Comme en Python, la borne finale d’une tranche est exclue : `1:3` prend les positions 1 et 2. Les positions négatives partent de la fin ; `-1` est la dernière position. `.iat` accède à une seule valeur par positions.

| Besoin | Étiquettes | Positions |
|---|---|---|
| Une sélection | `loc` | `iloc` |
| Une seule cellule | `at` | `iat` |
| Tranche | fin incluse ici | fin exclue |

Même avec un index entier, `.loc[2]` cherche l’étiquette 2, qui n’est pas nécessairement la troisième ligne.


In [ ]:
display(df_selection.iloc[:3, 1:3])
print(df_selection.iat[1, 0])


### À toi de jouer


1. Extrais la dernière ligne.
2. Extrais les lignes de positions 0 et 2, colonnes de positions 0 et 2.
3. Extrais les 2 premières lignes et les 2 premières colonnes avec des tranches.
4. Retrouve la valeur 2 obtenue précédemment en utilisant `.iat`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(df_selection.iloc[-1])
display(df_selection.iloc[[0, 2], [0, 2]])
display(df_selection.iloc[:2, :2])
print(df_selection.iat[2, 2])
```

</details>


<a id="notion-7"></a>

## 7. Filtrer : masques booléens et isin

**Repère dans le notebook original : cellules 42–46** (numérotation à partir de 1, textes compris).


Une comparaison comme `df["C"] > 1` produit une Series de booléens, appelée **masque**. `df[masque]` garde les lignes où il vaut `True`. `.isin([…])` teste l’appartenance à un ensemble de valeurs.

Pour combiner des conditions sur des Series, utilise `&` (et), `|` (ou), `~` (non), avec des parenthèses autour de chaque comparaison. Les opérateurs Python `and` et `or` ne combinent pas ces Series élément par élément.


In [ ]:
masque = df_selection["C"] > 1
display(masque, df_selection[masque])
display(df_selection[df_selection["C"].isin([0, 2])])


### À toi de jouer


1. Garde les lignes où `A >= 20` et `C < 3` (attendu : `Two`, `Three`).
2. Garde les lignes où `B` vaut 1 ou 4.
3. Exclue les lignes où `C` appartient à `[0, 2]`.
4. Pour les lignes où `A > 20`, n’affiche que la colonne `B` avec `.loc`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(df_selection[(df_selection["A"] >= 20) & (df_selection["C"] < 3)])
display(df_selection[df_selection["B"].isin([1, 4])])
display(df_selection[~df_selection["C"].isin([0, 2])])
display(df_selection.loc[df_selection["A"] > 20, "B"])
```

</details>


<a id="notion-8"></a>

## 8. Modifier des cellules et créer des colonnes

**Repère dans le notebook original : cellules 47–52** (numérotation à partir de 1, textes compris).


Les mêmes accesseurs servent à affecter une valeur : `df.loc[condition, "colonne"] = valeur`. Effectue une affectation en une seule sélection ; évite `df[condition]["colonne"] = ...`, qui peut modifier un objet intermédiaire.

Une valeur unique remplit la colonne ; une liste est affectée **par position** et doit avoir la bonne longueur. Une nouvelle étiquette de colonne crée une colonne. `.copy()` crée ici une copie de travail pour conserver les données de départ. Une opération comme `df["A"] * 2` agit sur toute la colonne : c’est une opération vectorisée.


In [ ]:
df_modif = df_selection.copy()
df_modif.iloc[0, 0] = 25
df_modif.loc[df_modif["C"] > 1, "B"] = 100
df_modif["origine"] = "capteur"
df_modif["actif"] = [True, False, False, True]
display(df_modif)


### À toi de jouer


1. Repars d’une copie de `df_selection` appelée `travail`.
2. Mets `A` à 99 pour la ligne `Two` avec `.at`.
3. Mets `B` à 0 lorsque `C >= 2`.
4. Crée `double_A`, puis `valide` avec les valeurs `True, True, False, True`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
travail = df_selection.copy()
travail.at["Two", "A"] = 99
travail.loc[travail["C"] >= 2, "B"] = 0
travail["double_A"] = travail["A"] * 2
travail["valide"] = [True, True, False, True]
display(travail)
```

</details>


<a id="notion-9"></a>

## 9. Comprendre l’alignement des Series

**Repère dans le notebook original : cellules 49–50** (numérotation à partir de 1, textes compris).


Quand on affecte une Series à une colonne, Pandas fait correspondre les **étiquettes de lignes**, pas les positions. Une étiquette absente de la Series donne `NaN` dans le DataFrame ; une étiquette présente seulement dans la Series n’ajoute pas automatiquement une ligne au DataFrame.

C’est pourquoi l’exemple original avec `Three, Four, One, five` ne remplit pas la ligne `Two`. Cette règle d’alignement intervient aussi dans de nombreuses opérations arithmétiques entre Series.


In [ ]:
df_align = pd.DataFrame(index=["One", "Two", "Three", "Four"])
df_align["A"] = pd.Series(range(4), index=["Three", "Four", "One", "five"])
display(df_align)  # One=2, Two=NaN, Three=0, Four=1.


### À toi de jouer


1. Crée un DataFrame indexé par `C1, C2, C3`.
2. Affecte-lui une colonne `mesure` depuis `pd.Series([30, 10], index=["C3", "C1"])`.
3. Prédis le résultat avant de l’afficher.
4. Ajoute `par_position` avec la liste `[30, 10, 20]`. Explique la différence.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
df_align_ex = pd.DataFrame(index=["C1", "C2", "C3"])
df_align_ex["mesure"] = pd.Series([30, 10], index=["C3", "C1"])
df_align_ex["par_position"] = [30, 10, 20]
display(df_align_ex)  # mesure : 10, NaN, 30 ; liste : 30, 10, 20.
```

</details>


<a id="notion-10"></a>

## 10. Définir, réinitialiser et réindexer un index

**Repère dans le notebook original : cellules 53–58** (numérotation à partir de 1, textes compris).


`set_index("colonne")` transforme une colonne en index (et la retire par défaut des colonnes). `reset_index()` remet un index `0, 1, 2…` et conserve l’ancien dans une colonne. `reset_index(drop=True)` abandonne l’ancien index.

Ces appels renvoient normalement un nouveau tableau : pour conserver le changement, affecte le résultat à une variable. `reindex(...)`, évoqué dans le texte original, permet de choisir de nouvelles étiquettes : réordonner, retirer ou ajouter des lignes. Les nouvelles lignes sont remplies de valeurs manquantes.


In [ ]:
df_index = pd.DataFrame({"id": ["C1", "C2"], "mesure": [10, 20]})
par_id = df_index.set_index("id")
display(par_id, par_id.reset_index(), par_id.reset_index(drop=True))
display(par_id.reindex(["C2", "C3", "C1"]))


### À toi de jouer


1. Mets `nom` en index d’une copie de `df_vue`.
2. Récupère le score de `B` avec `.loc` (attendu : **14**).
3. Réindexe dans l’ordre `D, A, E`. Pourquoi `E` contient-il une valeur manquante ?
4. Réinitialise l’index en conservant les noms dans une colonne.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
vue_indexee = df_vue.set_index("nom")
print(vue_indexee.loc["B", "score"])
vue_reordonnee = vue_indexee.reindex(["D", "A", "E"])
display(vue_reordonnee)  # E n’existait pas.
display(vue_reordonnee.reset_index())
```

</details>


<a id="notion-11"></a>

## 11. Valeurs manquantes : détecter, supprimer, remplir

**Repère dans le notebook original : cellules 58–70** (numérotation à partir de 1, textes compris).


`isna()` (ou `pd.isna`) détecte les valeurs manquantes ; ne teste pas `x == np.nan`. `dropna()` supprime les lignes contenant au moins une valeur manquante ; `axis=1` vise les colonnes ; `how="all"` exige que toutes les valeurs de la ligne soient manquantes. `subset=[…]` limite les colonnes examinées.

`fillna(valeur)` remplit les valeurs manquantes. Choisir une valeur de remplacement est une décision d’analyse : remplacer une mesure inconnue par zéro change sa signification. Sans réaffectation, ces méthodes ne modifient pas le tableau de départ. `isna().sum()` compte les manquants par colonne, car `True` compte pour 1.

Dans l’original, `% 17` est le reste de la division par 17 et le masque `(df % 17) == 0` vise les multiples de 17, y compris zéro.


In [ ]:
df_na = pd.DataFrame({"A": [1, np.nan, 3, np.nan], "B": [10, 20, np.nan, np.nan]})
display(df_na, df_na.isna(), df_na.isna().sum())
display(df_na.dropna(), df_na.dropna(axis=1), df_na.dropna(how="all"))
display(df_na.fillna(1000))


### À toi de jouer


1. Compte les valeurs manquantes de chaque colonne (attendu : **2 et 2**).
2. Supprime uniquement la ligne entièrement vide (il doit rester **3 lignes**).
3. Sur une copie, remplace les manquants de `A` par la moyenne de `A`, sans toucher `B`.
4. Crée les nombres de 0 à 11 dans un tableau 3 × 4, puis remplace les multiples de 5 par `np.nan`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
print(df_na.isna().sum())
display(df_na.dropna(how="all"))
na_corrige = df_na.copy()
na_corrige["A"] = na_corrige["A"].fillna(na_corrige["A"].mean())
display(na_corrige)  # A : 1, 2, 3, 2 ; B conserve ses manquants.
matrice = pd.DataFrame(np.arange(12, dtype=float).reshape(3, 4))
matrice[matrice % 5 == 0] = np.nan
display(matrice)  # 0, 5 et 10 remplacés.
```

</details>


<a id="notion-12"></a>

## 12. Détecter et supprimer les doublons

**Repère dans le notebook original : cellules 71–82** (numérotation à partir de 1, textes compris).


`duplicated()` renvoie un masque : la première occurrence est conservée (`False`), les répétitions valent `True`. `keep="last"` conserve la dernière ; `keep=False` marque toutes les occurrences des groupes répétés. `subset=` définit les colonnes qui déterminent l’identité d’une ligne. L’index n’est pas utilisé pour comparer les lignes.

`drop_duplicates()` supprime les lignes marquées. Deux lignes avec la même marque mais des notes différentes sont distinctes si l’on compare toutes les colonnes ; elles sont des doublons si l’on compare uniquement la marque. Il faut donc définir ce qu’est réellement un doublon dans ton problème.


In [ ]:
produits = pd.DataFrame({"brand": ["Yum Yum", "Yum Yum", "Indomie", "Indomie", "Indomie"],
                         "style": ["cup", "cup", "cup", "pack", "pack"],
                         "rating": [4, 4, 3.5, 15, 5]})
display(produits.duplicated(), produits.duplicated(keep="last"), produits.duplicated(keep=False))


### À toi de jouer


1. Affiche toutes les lignes qui appartiennent à un groupe de doublons exacts.
2. Supprime les doublons exacts (attendu : **4 lignes**).
3. Garde la dernière ligne par couple `brand, style` (attendu : **3 lignes**).
4. Garde seulement les marques qui n’apparaissent qu’une seule fois. Pourquoi le résultat est-il vide ?


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(produits[produits.duplicated(keep=False)])
display(produits.drop_duplicates())
display(produits.drop_duplicates(subset=["brand", "style"], keep="last"))
display(produits.drop_duplicates(subset=["brand"], keep=False))
# Les deux marques apparaissent chacune plusieurs fois.
```

</details>


<a id="notion-13"></a>

## 13. Concaténer des tableaux : concat, axis et join

**Repère dans le notebook original : cellules 83–92** (numérotation à partir de 1, textes compris).


`pd.concat([a, b])` empile les lignes (`axis=0`) et aligne les colonnes par leur nom. Par défaut, toutes les colonnes sont conservées (`join="outer"`) ; les cases sans correspondance deviennent manquantes. Les index d’origine sont conservés, même s’ils se répètent. `ignore_index=True` recrée les étiquettes **sur l’axe concaténé**.

Avec `axis=1`, on juxtapose les colonnes en alignant les index de lignes : `ignore_index=True` renumérote alors les **colonnes**, pas les lignes. `join="inner"` garde seulement les étiquettes communes sur l’autre axe (colonnes pour `axis=0`, lignes pour `axis=1`).


In [ ]:
lot_a = pd.DataFrame({"A": [1, 2], "B": [10, 20]}, index=["x", "y"])
lot_b = pd.DataFrame({"A": [3], "C": [30]}, index=["z"])
display(pd.concat([lot_a, lot_b], ignore_index=True))
col_c = pd.DataFrame({"C": [100, 200]}, index=["y", "z"])
display(pd.concat([lot_a, col_c], axis=1))


### À toi de jouer


1. Empile `lot_a` et `lot_b` en gardant seulement les colonnes communes et un nouvel index (attendu : `A`, valeurs **1, 2, 3**).
2. Juxtapose `lot_a` et `col_c` en gardant seulement les lignes communes (attendu : ligne **y**, valeurs **2, 20, 100**).
3. Refais la juxtaposition avec `ignore_index=True`. Quel axe est renommé ?


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(pd.concat([lot_a, lot_b], join="inner", ignore_index=True))
display(pd.concat([lot_a, col_c], axis=1, join="inner"))
display(pd.concat([lot_a, col_c], axis=1, join="inner", ignore_index=True))
# Les colonnes deviennent 0, 1, 2 ; l’index de ligne reste y.
```

</details>


<a id="notion-14"></a>

## 14. Joindre sur une clé : merge et multiplicité

**Repère dans le notebook original : cellules 93–96** (numérotation à partir de 1, textes compris).


`pd.merge(gauche, droite, on="clé")` associe des lignes selon les **valeurs d’une colonne**, même si elles ne sont pas dans le même ordre. Par défaut, `how="inner"` conserve les clés présentes des deux côtés ; `how="left"` conserve toutes les lignes de gauche ; `how="outer"` conserve toutes les clés.

Si une même clé apparaît 3 fois à gauche et 3 fois à droite, la jointure produit **3 × 3 = 9 combinaisons** pour cette clé. Ce ne sont pas nécessairement 9 lignes identiques. `validate="one_to_one"` ou `"many_to_one"` permet de vérifier la relation attendue au lieu de multiplier silencieusement les lignes.


In [ ]:
mesures = pd.DataFrame({"id": ["C1", "C2", "C3"], "valeur": [10, 20, 30]})
zones = pd.DataFrame({"id": ["C2", "C1"], "zone": ["sud", "nord"]})
display(pd.merge(mesures, zones, on="id"))


### À toi de jouer


1. Joins `mesures` et `zones` en conservant `C3`. Quelle valeur aura sa zone ?
2. Ajoute une vérification d’unicité des clés des deux côtés.
3. Crée 2 lignes de clé `X` à gauche et 3 à droite ; vérifie que leur jointure a **6 lignes**.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(mesures.merge(zones, on="id", how="left", validate="one_to_one"))
# La zone de C3 est manquante.
gauche = pd.DataFrame({"cle": ["X", "X"], "a": [1, 2]})
droite = pd.DataFrame({"cle": ["X", "X", "X"], "b": [10, 20, 30]})
combinaisons = gauche.merge(droite, on="cle")
display(combinaisons)
print(len(combinaisons))  # 6.
```

</details>


<a id="notion-15"></a>

## 15. Statistiques et sens des axes

**Repère dans le notebook original : cellules 97–99** (numérotation à partir de 1, textes compris).


`min`, `max`, `sum`, `mean`, `std` et `var` résument les valeurs. Sur un DataFrame, `axis=0` réduit les lignes et donne **un résultat par colonne** ; `axis=1` réduit les colonnes et donne **un résultat par ligne**. Dans l’original, `func(1)` équivaut ici à `func(axis=1)`.

Les méthodes présentées ignorent les valeurs manquantes par défaut (`skipna=True`). L’écart-type `std()` et la variance `var()` utilisent par défaut `ddof=1` (division par N−1), contrairement au défaut de NumPy qui utilise `ddof=0`. Une seule valeur valide ne suffit donc pas pour calculer cet écart-type. `sum()` peut renvoyer zéro pour une série entièrement manquante ; `sum(min_count=1)` exige au moins une valeur valide.


In [ ]:
stats = pd.DataFrame({"A": [1., 3., 5.], "B": [2., np.nan, 8.]})
for nom in ["min", "max", "sum", "std", "var"]:
    print(nom, "par colonne :", getattr(stats, nom)(axis=0).to_dict())
    print(nom, "par ligne :", getattr(stats, nom)(axis=1).to_list())


### À toi de jouer


1. Calcule la moyenne de chaque colonne (attendu : **A=3, B=5**).
2. Calcule la somme de chaque ligne (attendu : **3, 3, 13**).
3. Compare `std()` et `std(ddof=0)` pour `A`.
4. Explique pourquoi la moyenne de la deuxième ligne vaut 3 et non 1.5.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
print(stats.mean(axis=0))
print(stats.sum(axis=1))
print(stats["A"].std(), stats["A"].std(ddof=0))  # 2 et environ 1.633.
print(stats.mean(axis=1))  # Le NaN n’entre pas dans le dénominateur.
```

</details>


<a id="notion-16"></a>

## 16. Fonctions, apply, map et opérations vectorisées

**Repère dans le notebook original : cellules 100–109** (numérotation à partir de 1, textes compris).


`def` définit une fonction réutilisable et `return` renvoie son résultat. Une boucle appelle explicitement la fonction pour chaque valeur. `Series.apply(f)` applique ici `f` aux éléments de la Series. `Series.map(f)` transforme aussi les éléments et accepte un dictionnaire de correspondance.

`DataFrame.apply(f, axis=0)` passe chaque **colonne** à `f` ; `axis=1` passe chaque **ligne** sous forme de Series. Le type du résultat dépend de ce que renvoie la fonction : ce n’est pas toujours un DataFrame.

Pour une formule simple, préfère une opération vectorisée (`serie ** 2`, `serie + 1`) ou une méthode dédiée (`serie.str.upper()`). `apply()` peut être utile, mais ne garantit pas un gain de vitesse par rapport à une boucle.


In [ ]:
def carre(x):
    return x * x

fonctions = pd.DataFrame({"nombre": [1, 2, 3], "lettre": ["a", "b", "c"]})
fonctions["carre"] = fonctions["nombre"].apply(carre)
fonctions["majuscule"] = fonctions["lettre"].map(str.upper)
display(fonctions)
print(fonctions["nombre"] ** 2)


### À toi de jouer


1. Définis `ajouter_un` puis applique-la à `nombre`.
2. Obtiens les mêmes valeurs sans `apply`.
3. Convertis `lettre` en majuscules avec `.str.upper()`.
4. Utilise `DataFrame.apply(axis=1)` pour calculer `nombre + carre` ligne par ligne (attendu : **2, 6, 12**).


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
def ajouter_un(x):
    return x + 1

print(fonctions["nombre"].apply(ajouter_un))
print(fonctions["nombre"] + 1)
print(fonctions["lettre"].str.upper())
print(fonctions.apply(lambda ligne: ligne["nombre"] + ligne["carre"], axis=1))
```

</details>


<a id="notion-17"></a>

## 17. Lambda, modulo et dictionnaires de correspondance

**Repère dans le notebook original : cellules 110–111** (numérotation à partir de 1, textes compris).


`lambda x: expression` définit une petite fonction anonyme : `lambda x: x % 3` renvoie le reste de la division par 3. Une lambda convient aux expressions courtes ; utilise `def` si la logique demande plusieurs étapes.

`dictionnaire[x]` recherche une valeur et lève une erreur si la clé est absente. Pour traduire les valeurs d’une Series, `.map(dictionnaire)` est souvent plus direct ; une clé absente donne alors une valeur manquante. Une expression conditionnelle a la forme `valeur_si_vrai if condition else valeur_si_faux`.


In [ ]:
lettres = pd.Series(["a", "b", "c", "z"])
correspondance = {"a": "anaconda", "b": "boa", "c": "cobra"}
display(lettres.map(correspondance))
display(pd.Series([1, 2, 3, 4, 5]).apply(lambda x: x % 3))


### À toi de jouer


1. Classe les nombres de 1 à 6 en `pair` ou `impair` à l’aide d’une lambda.
2. Traduis `a, b, c, z` avec `correspondance`, puis remplace la traduction manquante par `inconnu`.
3. Écris le calcul du reste modulo 3 sans `apply`.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
nombres = pd.Series(range(1, 7))
print(nombres.apply(lambda x: "pair" if x % 2 == 0 else "impair"))
print(lettres.map(correspondance).fillna("inconnu"))
print(nombres % 3)
```

</details>


<a id="notion-18"></a>

## 18. Lire le CSV et contrôler les données

**Repère dans le notebook original : cellules 113–117** (numérotation à partir de 1, textes compris).


`pd.read_csv(chemin, sep=";")` lit un fichier dont les champs sont séparés par des points-virgules. Avec le mauvais séparateur, tu peux obtenir une seule grande colonne. Un chemin relatif est interprété depuis le **dossier de travail du noyau**, pas nécessairement depuis le fichier notebook.

Le CSV voisin contient `timestamp_utc`, deux colonnes en `kWh` (énergie active) et deux en `kvarh` (énergie réactive). Ne mélange pas ces unités. Vérifie le nombre de lignes, les types, les valeurs manquantes et les premières lignes avant de calculer. La recherche ci-dessous couvre un noyau lancé depuis `personal_folder` ou depuis la racine du projet.


In [ ]:
chemin_csv = Path("pandas_timeseries_test.csv")
if not chemin_csv.exists():
    chemin_csv = Path("personal_folder") / "pandas_timeseries_test.csv"
if not chemin_csv.exists():
    raise FileNotFoundError("Place le dossier de travail dans personal_folder ou à la racine du projet.")

smartmeter_brut = pd.read_csv(chemin_csv, sep=";")
display(smartmeter_brut.head())
smartmeter_brut.info()
print(smartmeter_brut.shape)


### À toi de jouer


1. Affiche les 3 dernières lignes, les noms de colonnes et le nombre de valeurs manquantes.
2. Copie les colonnes de date et les deux colonnes en kWh dans `energie_brute`.
3. Vérifie le nombre de lignes (le fichier fourni en contient **70 176**) et calcule les totaux des deux colonnes en kWh.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(smartmeter_brut.tail(3))
print(smartmeter_brut.columns)
print(smartmeter_brut.isna().sum())
energie_brute = smartmeter_brut[["timestamp_utc", "kWh_to_installation", "kWh_to_grid"]].copy()
print(len(energie_brute))
print(energie_brute[["kWh_to_installation", "kWh_to_grid"]].sum())
```

</details>


<a id="notion-19"></a>

## 19. Dates, fuseaux horaires et pas de temps

**Repère dans le notebook original : cellules 118–119** (numérotation à partir de 1, textes compris).


`pd.to_datetime(...)` convertit des chaînes en dates manipulables. `utc=True` normalise ici les instants en UTC et garde un fuseau explicite. `sort_values` classe chronologiquement ; `diff()` mesure l’écart entre deux instants successifs. Sa première valeur est manquante puisqu’il n’y a pas d’instant précédent.

Le CSV fourni va du **31 décembre 2022 à 23:00 UTC au 31 décembre 2024 à 22:45 UTC**, avec un pas de **15 minutes**. Cela correspond aux années locales 2023 et 2024 en Suisse. `dt.tz_convert("Europe/Zurich")` change l’affichage dans le fuseau local sans changer les instants. L’heure locale tient compte des changements d’heure ; une série UTC régulière reste plus simple à contrôler.

Pour filtrer une année, utilise un intervalle semi-ouvert : début inclus, début de l’année suivante exclu. Précise toujours le fuseau de cette année.


In [ ]:
smartmeter = smartmeter_brut.copy()
smartmeter["timestamp_utc"] = pd.to_datetime(smartmeter["timestamp_utc"], utc=True)
smartmeter = smartmeter.sort_values("timestamp_utc").reset_index(drop=True)
print(smartmeter["timestamp_utc"].min(), smartmeter["timestamp_utc"].max())
print(smartmeter["timestamp_utc"].diff().value_counts())
print(smartmeter["timestamp_utc"].duplicated().sum())
display(smartmeter["timestamp_utc"].dt.tz_convert("Europe/Zurich").head())


### À toi de jouer


1. Contrôle que tous les écarts non manquants valent 15 minutes et qu’aucun horodatage n’est répété.
2. Sélectionne l’année **2024 en Europe/Zurich** dans `smartmeter_2024`.
3. Vérifie que tu obtiens **35 136 lignes** : 366 jours × 24 heures × 4 mesures/heure.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
temps = smartmeter["timestamp_utc"]
print(temps.diff().dropna().eq(pd.Timedelta(minutes=15)).all())
print(temps.duplicated().sum())
debut_local = pd.Timestamp("2024-01-01", tz="Europe/Zurich")
fin_locale = pd.Timestamp("2025-01-01", tz="Europe/Zurich")
smartmeter_2024 = smartmeter.loc[(temps >= debut_local) & (temps < fin_locale)].copy()
print(len(smartmeter_2024))  # 35136, année bissextile.
```

</details>


<a id="notion-20"></a>

## 20. Autoconsommation : minimum et taux global

**Repère dans le notebook original : cellules 120–123** (numérotation à partir de 1, textes compris).


Le code original calcule, à chaque intervalle, `sc = min(consommation, production)`. Dans un modèle sans stockage, c’est l’énergie produite et consommée simultanément. Le **taux d’autoconsommation** est `somme(sc) / somme(production) × 100`. Le **taux d’autonomie** utilise la consommation au dénominateur. Le taux global n’est généralement pas la moyenne des pourcentages de chaque intervalle.

**Hypothèse pédagogique à vérifier sur le compteur :** l’original traite `kWh_to_installation` comme la consommation totale et `kWh_to_grid` comme la production totale. Les noms pourraient aussi désigner des flux importés/exportés du réseau. Le CSV seul ne permet pas de trancher. S’il s’agit de flux réseau, les deux colonnes seules ne suffisent pas à reconstituer l’autoconsommation : il faut aussi connaître la production totale ou la consommation totale. Les calculs ci-dessous reproduisent donc **le modèle du notebook**, pas une validation physique de ces colonnes.

Les mesures sont supposées non négatives, synchronisées et sans valeurs manquantes. Si la production totale vaut zéro, son taux d’autoconsommation est indéfini : on renvoie `np.nan`.


In [ ]:
energie = smartmeter[["timestamp_utc", "kWh_to_installation", "kWh_to_grid"]].rename(
    columns={"kWh_to_installation": "consommation_kWh", "kWh_to_grid": "production_kWh"}
).copy()
energie["sc"] = energie.apply(
    lambda ligne: ligne["production_kWh"] if ligne["consommation_kWh"] > ligne["production_kWh"]
    else ligne["consommation_kWh"], axis=1
)
production_totale = energie["production_kWh"].sum()
taux_sc = 100 * energie["sc"].sum() / production_totale if production_totale > 0 else np.nan
print("Taux d’autoconsommation du modèle (%) :", taux_sc)


### À toi de jouer


1. Crée un tableau avec consommation `[2, 4, 1]` et production `[3, 1, 2]` en kWh.
2. Calcule `sc` par ligne avec `apply`, puis avec `np.minimum`.
3. Retrouve **[2, 1, 1]**, puis un taux d’autoconsommation de **66.67 %** et un taux d’autonomie de **57.14 %** environ.
4. Vérifie que `sc` ne dépasse jamais la consommation ni la production.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
petit = pd.DataFrame({"consommation": [2, 4, 1], "production": [3, 1, 2]})
petit["sc_apply"] = petit.apply(lambda ligne: min(ligne["consommation"], ligne["production"]), axis=1)
petit["sc"] = np.minimum(petit["consommation"], petit["production"])
print(petit["sc"].to_list())
print(100 * petit["sc"].sum() / petit["production"].sum())
print(100 * petit["sc"].sum() / petit["consommation"].sum())
print((petit["sc"] <= petit["consommation"]).all(), (petit["sc"] <= petit["production"]).all())
```

</details>


<a id="notion-21"></a>

## 21. Tracer des courbes avec Plotly

**Repère dans le notebook original : cellules 124–125 et 133–134** (numérotation à partir de 1, textes compris).


`px.line()` crée une figure. `fig.add_scatter(x=..., y=..., mode="lines", name=...)` ajoute une courbe ; `name` définit sa légende. `update_layout` règle les titres. `fig.show()` affiche la figure dans un environnement compatible ; dans ce notebook, `display(fig)` l’affiche également.

Les horodatages doivent être triés et les axes doivent donner les unités. Les valeurs du CSV sont des **kWh par intervalle de 15 minutes**, pas des kW. Le tracé original ajoute seulement consommation et production malgré son titre « Plot SC » : nous ajoutons explicitement la courbe d’autoconsommation. Afficher un jour facilite la lecture.


In [ ]:
jour = energie.head(96)  # 96 intervalles consécutifs = 24 h.
fig = px.line()
for colonne, nom in [("consommation_kWh", "Consommation"), ("production_kWh", "Production"), ("sc", "Autoconsommation")]:
    fig.add_scatter(x=jour["timestamp_utc"], y=jour[colonne], name=nom, mode="lines")
fig.update_layout(title="Énergie sur 24 heures — modèle pédagogique", xaxis_title="Date (UTC)",
                  yaxis_title="Énergie par intervalle de 15 min [kWh]")
display(fig)


### À toi de jouer


1. Trace les mêmes trois courbes sur les **7 premiers jours** (7 × 96 lignes).
2. Donne un titre à la figure et conserve des unités explicites.
3. Repère les moments où l’autoconsommation est nulle. Dans ce modèle, que peut-on en déduire ?


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
semaine = energie.head(7 * 96)
fig_semaine = px.line(semaine, x="timestamp_utc", y=["consommation_kWh", "production_kWh", "sc"],
                      title="Consommation, production et autoconsommation — 7 jours",
                      labels={"timestamp_utc": "Date (UTC)", "value": "Énergie par 15 min [kWh]", "variable": "Grandeur"})
display(fig_semaine)
# Pour des valeurs non négatives, sc=0 signifie production=0 ou consommation=0 (ou les deux).
```

</details>


<a id="notion-22"></a>

## 22. date_range et conversion d’une production de 50 W

**Repère dans le notebook original : cellules 126–127** (numérotation à partir de 1, textes compris).


`pd.date_range(debut, fin, freq="15min", inclusive="left")` crée un `DatetimeIndex` régulier ; la borne finale est exclue. Pour couvrir toute l’année locale 2024, on utilise le 1er janvier 2024 et le 1er janvier 2025 en `Europe/Zurich`, puis on convertit les instants en UTC pour la jointure.

**Une puissance n’est pas une énergie.** `énergie [kWh] = puissance [W] / 1000 × durée [h]`. Une production constante de 50 W vaut 0.05 kW et apporte **0.0125 kWh par quart d’heure**. Sur les 366 jours de 2024, elle représente **439.2 kWh**, si elle fonctionne sans interruption.

L’original mentionne à la fois une production et une consommation constante. On retient ici une production hydroélectrique supplémentaire, conformément aux variables `hydro_production_df` et `sc_hydro`.


In [ ]:
debut_2024 = pd.Timestamp("2024-01-01", tz="Europe/Zurich")
fin_2024 = pd.Timestamp("2025-01-01", tz="Europe/Zurich")
index_2024 = pd.date_range(debut_2024, fin_2024, freq="15min", inclusive="left").tz_convert("UTC")
hydro_production_df = pd.DataFrame({"timestamp_utc": index_2024, "hydro_kWh": 50 / 1000 * (15 / 60)})
print(len(hydro_production_df), hydro_production_df["hydro_kWh"].sum())
display(hydro_production_df.head())


### À toi de jouer


1. Crée une série de production constante de **80 W**, toutes les **30 minutes**, pour la journée du 1er février 2024 en UTC.
2. Vérifie que tu obtiens **48 intervalles**, **0.04 kWh par intervalle** et **1.92 kWh au total**.
3. Explique pourquoi saisir `80` directement dans une colonne en kWh serait incorrect.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
instants = pd.date_range("2024-02-01", "2024-02-02", freq="30min", inclusive="left", tz="UTC")
production_80w = pd.DataFrame({"timestamp_utc": instants, "energie_kWh": 80 / 1000 * (30 / 60)})
print(len(production_80w), production_80w["energie_kWh"].iloc[0], production_80w["energie_kWh"].sum())
# 80 est une puissance en W ; 0.04 est l’énergie en kWh produite en une demi-heure.
```

</details>


<a id="notion-23"></a>

## 23. Projet de synthèse : joindre et recalculer avec l’hydro

**Repère dans le notebook original : cellules 128–134** (numérotation à partir de 1, textes compris).


Ce projet complète la fin du notebook original. Il combine filtre temporel, jointure, contrôle des manquants, création de colonnes, statistiques et tracé.

Pour comparer les scénarios, on utilise **la même année 2024 locale** avant et après l’ajout. On joint sur les horodatages UTC avec `validate="one_to_one"`, puis on vérifie la couverture : l’unicité seule ne garantit pas que toutes les dates correspondent. Il ne faut pas masquer des erreurs de dates en remplaçant automatiquement les valeurs manquantes par zéro.

La production combinée est `production_initiale + hydro`. L’autoconsommation combinée vaut `min(consommation, production_combinée)`. La quantité autoconsommée ne peut pas diminuer si on ajoute une production non négative, mais **son pourcentage peut diminuer** si la production augmente davantage que l’énergie autoconsommée. On distingue donc les kWh supplémentaires et les points de pourcentage.


In [ ]:
energie_2024 = energie.loc[
    (energie["timestamp_utc"] >= debut_2024) & (energie["timestamp_utc"] < fin_2024)
].copy()
print("Lignes compteur / hydro :", len(energie_2024), len(hydro_production_df))


### À toi de jouer


1. Joins `energie_2024` et `hydro_production_df` en gardant les lignes compteur ; impose une relation un-à-un.
2. Vérifie le nombre de lignes et l’absence de `hydro_kWh` manquants.
3. Crée `production_totale_kWh` puis `sc_hydro` sans `apply`.
4. Calcule les taux d’autoconsommation avant et après et le gain en kWh autoconsommés.
5. Vérifie que `sc_hydro >= sc`, puis trace `sc` et `sc_hydro` sur la première semaine de l’année locale.
6. Explique les limites physiques du résultat en relisant l’hypothèse de la partie 20.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
bilan = energie_2024.merge(hydro_production_df, on="timestamp_utc", how="left", validate="one_to_one")
assert len(bilan) == len(energie_2024) == 35136
assert bilan["hydro_kWh"].notna().all(), "Certains horodatages ne correspondent pas."
bilan["production_totale_kWh"] = bilan["production_kWh"] + bilan["hydro_kWh"]
bilan["sc_hydro"] = np.minimum(bilan["consommation_kWh"], bilan["production_totale_kWh"])

def taux_autoconsommation(sc, production):
    total = production.sum()
    return 100 * sc.sum() / total if total > 0 else np.nan

avant = taux_autoconsommation(bilan["sc"], bilan["production_kWh"])
apres = taux_autoconsommation(bilan["sc_hydro"], bilan["production_totale_kWh"])
print(f"Avant : {avant:.2f} % ; après : {apres:.2f} %")
print(f"Variation : {apres - avant:.2f} points de pourcentage")
print(f"Gain d’énergie autoconsommée : {bilan['sc_hydro'].sum() - bilan['sc'].sum():.2f} kWh")
assert (bilan["sc_hydro"] >= bilan["sc"]).all()
assert (bilan["sc_hydro"] <= bilan["consommation_kWh"]).all()
fig_bilan = px.line(bilan.head(7 * 96), x="timestamp_utc", y=["sc", "sc_hydro"],
                    title="Autoconsommation avant et après ajout de 50 W — modèle pédagogique",
                    labels={"timestamp_utc": "Date (UTC)", "value": "Énergie par 15 min [kWh]", "variable": "Scénario"})
display(fig_bilan)
# Résultats physiques valables uniquement si les colonnes sont bien les totaux consommation/production.
```

</details>


<a id="notion-24"></a>

## 24. Pour aller plus loin : groupby et tableau croisé

**Repère dans le notebook original : cellules 1 (notions citées seulement dans l’introduction)** (numérotation à partir de 1, textes compris).


L’introduction évoque les regroupements et les tableaux croisés sans les développer. Ce complément permet de les essayer : `groupby("colonne")["mesure"].sum()` regroupe les lignes par catégorie, puis additionne les mesures. `pivot_table(index=..., columns=..., values=..., aggfunc=...)` répartit les groupes en lignes et en colonnes, comme un tableau croisé dans un tableur. Il faut préciser l’agrégation souhaitée : plusieurs observations peuvent appartenir à la même case.

Un index à plusieurs niveaux (**MultiIndex**, seulement cité comme lecture avancée dans l’original) peut représenter les résultats d’un regroupement selon plusieurs clés.


In [ ]:
releves = pd.DataFrame({"jour": ["lun", "lun", "mar", "mar"], "zone": ["nord", "sud", "nord", "sud"],
                        "kWh": [2, 3, 4, 1]})
display(releves.groupby("zone")["kWh"].sum())
display(releves.pivot_table(index="jour", columns="zone", values="kWh", aggfunc="sum"))


### À toi de jouer


1. Calcule l’énergie totale par jour (attendu : **5 kWh chaque jour**).
2. Crée un tableau croisé avec les zones en lignes et les jours en colonnes.
3. Regroupe par `jour` et `zone` puis observe l’index du résultat.


In [ ]:
# Écris ta solution ici.


<details>
<summary>Afficher le corrigé après avoir essayé</summary>

```python
display(releves.groupby("jour")["kWh"].sum())
display(releves.pivot_table(index="zone", columns="jour", values="kWh", aggfunc="sum"))
par_jour_zone = releves.groupby(["jour", "zone"])["kWh"].sum()
display(par_jour_zone)
print(type(par_jour_zone.index))  # MultiIndex.
```

</details>


## Fiche de révision

| Objectif | Outil principal | Point à surveiller |
|---|---|---|
| Créer une colonne de valeurs | `pd.Series` | Valeurs et index |
| Créer un tableau | `pd.DataFrame` | Longueurs et types |
| Examiner | `head`, `tail`, `info`, `describe`, `dtypes` | Structure ou statistiques ? |
| Sélectionner | `[]`, `loc`, `iloc`, `at`, `iat` | Étiquette ou position ? |
| Filtrer | comparaison, `isin`, `&`, `\|`, `~` | Parenthèses autour des conditions |
| Modifier | `df.loc[...] = ...` | Une seule affectation |
| Affecter une Series | `df["col"] = serie` | Alignement par étiquette |
| Changer l’index | `set_index`, `reset_index`, `reindex` | Conserver le résultat |
| Traiter les manquants | `isna`, `dropna`, `fillna` | Inconnu ≠ zéro |
| Traiter les doublons | `duplicated`, `drop_duplicates` | `subset` et `keep` |
| Empiler ou juxtaposer | `concat` | Axe et alignement |
| Joindre par valeur | `merge` | Multiplicité des clés |
| Résumer | `sum`, `mean`, `min`, `max`, `std`, `var` | Axe, NaN et `ddof` |
| Transformer | `apply`, `map`, opérations vectorisées | Élément, ligne ou colonne ? |
| Lire un CSV | `read_csv` | Chemin et séparateur |
| Manipuler le temps | `to_datetime`, `date_range` | Fuseau et pas temporel |
| Tracer | `px.line`, `add_scatter`, `update_layout` | Unités et ordre temporel |
| Calculer une énergie | `W / 1000 × heures` | kW ≠ kWh |

**Autoévaluation :** sais-tu expliquer pourquoi une affectation crée un NaN, pourquoi une jointure multiplie les lignes, pourquoi `axis=1` change le calcul, et pourquoi 50 W deviennent 0.0125 kWh par quart d’heure ? Refais les exercices correspondants sans consulter le corrigé.

Les liens de référence de l’original se trouvent dans ses cellules 1, 11, 22, 26, 34, 58, 73, 81, 84, 93, 100 et 112. Ce support est construit à partir du notebook local et du CSV fourni ; les compléments pédagogiques sont identifiés dans le texte.
